# Portfolio Construction and Optimization Framework

## Overview

This project implements and compares three widely used portfolio optimization methodologies in institutional asset management. The objective is to construct efficient portfolios under different assumptions about return distributions, investor views, and risk allocation frameworks.

The models included are:

- **Markowitz Mean-Variance Optimization**  
  Classical framework that derives optimal portfolio weights by minimizing variance for a given expected return (or maximizing return for a given level of risk). It serves as the foundational benchmark model.

- **Black-Litterman / Litterman Framework**  
  A Bayesian approach that combines market equilibrium returns with investor views to produce more stable and realistic expected returns. It addresses estimation error sensitivity in traditional mean-variance optimization.

- **Hierarchical Risk Parity (HRP)**  
  A modern machine learning-inspired allocation method that builds portfolios based on hierarchical clustering of assets and distributes risk without relying on covariance matrix inversion or expected returns.

## Objective

The framework is designed to:

- Compare different portfolio construction methodologies under consistent inputs  
- Evaluate stability, diversification, and risk-adjusted performance  
- Reduce estimation error and improve robustness of allocations  
- Provide an institutional-grade toolkit for portfolio design and analysis  

## Output

Each model produces:

- Portfolio weights  
- Risk and return statistics  
- Comparative performance metrics  
- (Where applicable) efficient frontier / allocation hierarchy  

This allows for direct comparison between classical optimization, Bayesian adjustment, and data-driven risk allocation approaches.

# 1. Markowitz Mean-Variance Optimization

## Overview

This section implements a classical **Markowitz Mean-Variance Optimization** framework for portfolio construction. The model serves as the foundational benchmark in modern portfolio theory and is widely used in institutional asset management as a first-pass allocation engine.

The optimizer dynamically retrieves market data from Yahoo Finance and constructs efficient portfolios based on historical returns and covariance structure.

## Methodology

The model is based on the following inputs:

- Historical price data (Yahoo Finance)
- Daily log returns (annualized using 252 trading days)
- Expected returns estimated as historical mean returns
- Risk measured through the covariance matrix of returns

Portfolio construction is performed by solving a constrained optimization problem:

- **Objective functions:**
  - Maximize Sharpe ratio (risk-adjusted return)
  - Minimize portfolio volatility
  - Target a specific expected return (efficient frontier generation)

- **Constraints:**
  - Full investment constraint (weights sum to 1)
  - No short-selling (weights bounded between 0 and 1)

## Outputs

The model generates:

- Optimal portfolio weights under different objectives
- Portfolio risk/return metrics (expected return, volatility, Sharpe ratio)
- Efficient frontier visualization
- Strategy comparison (equal weight vs optimized portfolios)

## Interpretation

This framework provides a baseline allocation model that:

- Maximizes risk-adjusted performance under historical assumptions  
- Highlights diversification benefits across correlated assets  
- Serves as a benchmark for more advanced allocation techniques (Black-Litterman and HRP)

## Limitations

While foundational, the Markowitz framework has well-known limitations:

- Highly sensitive to estimation error in expected returns  
- Instability of weights under small input changes  
- Assumes Gaussian return distributions  
- Does not incorporate investor views or hierarchical structure in risk

In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from datetime import datetime, timedelta

class MarkowitzPortfolioOptimizer:
    """
    Markowitz Mean-Variance Portfolio Optimization Model.
    Automatically fetches data from Yahoo Finance and optimizes portfolio allocation.
    """
    
    def __init__(self, tickers, period='3y', risk_free_rate=0.04):
        """
        Initialize the portfolio optimizer.
        
        Parameters:
        - tickers: List of stock ticker symbols (e.g., ['AAPL', 'MSFT', 'GOOGL'])
        - period: Historical data period ('1y', '2y', '3y', '5y', '10y', 'max')
        - risk_free_rate: Annual risk-free rate (default 4%)
        """
        self.tickers = tickers
        self.period = period
        self.risk_free_rate = risk_free_rate
        self.n_assets = len(tickers)
        
        # Data storage
        self.prices = None
        self.returns = None
        self.mean_returns = None
        self.cov_matrix = None
        self.optimal_weights = None
        
        print(f"Initializing portfolio with {self.n_assets} assets: {', '.join(tickers)}")
        
    def fetch_data(self):
        """Fetch historical price data from Yahoo Finance."""
        print(f"\nFetching {self.period} of historical data from Yahoo Finance...")
        
        try:
            # Download data
            data = yf.download(self.tickers, period=self.period, progress=False, auto_adjust=True)
            
            # Handle single vs multiple tickers
            if len(self.tickers) == 1:
                # Single ticker: data is a simple DataFrame
                self.prices = data[['Close']].copy()
                self.prices.columns = self.tickers
            else:
                # Multiple tickers: extract Close prices
                if 'Close' in data.columns:
                    self.prices = data['Close'].copy()
                else:
                    # Fallback if structure is different
                    self.prices = data.copy()
            
            # Remove any NaN values
            self.prices = self.prices.dropna()
            
            # Calculate daily returns
            self.returns = self.prices.pct_change().dropna()
            
            # Calculate annualized mean returns (252 trading days)
            self.mean_returns = self.returns.mean() * 252
            
            # Calculate annualized covariance matrix
            self.cov_matrix = self.returns.cov() * 252
            
            print(f"✓ Successfully fetched data for {len(self.prices)} trading days")
            print(f"  Date range: {self.prices.index[0].date()} to {self.prices.index[-1].date()}")
            
            return True
            
        except Exception as e:
            print(f"✗ Error fetching data: {e}")
            import traceback
            traceback.print_exc()
            return False
    
    def portfolio_performance(self, weights):
        """
        Calculate portfolio performance metrics.
        
        Returns: (expected_return, volatility, sharpe_ratio)
        """
        weights = np.array(weights)
        
        # Portfolio return
        portfolio_return = np.sum(self.mean_returns * weights)
        
        # Portfolio volatility (risk)
        portfolio_volatility = np.sqrt(np.dot(weights.T, np.dot(self.cov_matrix, weights)))
        
        # Sharpe ratio
        sharpe_ratio = (portfolio_return - self.risk_free_rate) / portfolio_volatility
        
        return portfolio_return, portfolio_volatility, sharpe_ratio
    
    def negative_sharpe(self, weights):
        """Objective function to minimize (negative Sharpe ratio)."""
        return -self.portfolio_performance(weights)[2]
    
    def optimize_portfolio(self, target='max_sharpe', target_return=None):
        """
        Optimize portfolio allocation.
        
        Parameters:
        - target: 'max_sharpe' (default), 'min_volatility', or 'target_return'
        - target_return: Required if target='target_return'
        
        Returns: Optimal weights
        """
        # Constraints: weights sum to 1
        constraints = [{'type': 'eq', 'fun': lambda x: np.sum(x) - 1}]
        
        # Bounds: weights between 0 and 1 (no short selling)
        bounds = tuple((0, 1) for _ in range(self.n_assets))
        
        # Initial guess: equal weights
        initial_weights = np.array([1/self.n_assets] * self.n_assets)
        
        if target == 'max_sharpe':
            # Maximize Sharpe ratio
            result = minimize(
                self.negative_sharpe,
                initial_weights,
                method='SLSQP',
                bounds=bounds,
                constraints=constraints
            )
            
        elif target == 'min_volatility':
            # Minimize volatility
            result = minimize(
                lambda w: self.portfolio_performance(w)[1],
                initial_weights,
                method='SLSQP',
                bounds=bounds,
                constraints=constraints
            )
            
        elif target == 'target_return':
            if target_return is None:
                raise ValueError("target_return must be specified")
            
            # Add return constraint
            constraints.append({
                'type': 'eq',
                'fun': lambda w: self.portfolio_performance(w)[0] - target_return
            })
            
            result = minimize(
                lambda w: self.portfolio_performance(w)[1],
                initial_weights,
                method='SLSQP',
                bounds=bounds,
                constraints=constraints
            )
        
        if result.success:
            self.optimal_weights = result.x
            return result.x
        else:
            print(f"Optimization failed: {result.message}")
            return None
    
    def generate_efficient_frontier(self, n_portfolios=100):
        """
        Generate efficient frontier by optimizing for different target returns.
        
        Returns: DataFrame with frontier portfolios
        """
        # Get range of possible returns
        min_return = self.mean_returns.min()
        max_return = self.mean_returns.max()
        target_returns = np.linspace(min_return, max_return, n_portfolios)
        
        frontier_returns = []
        frontier_volatilities = []
        frontier_sharpes = []
        
        print("\nGenerating efficient frontier...")
        
        for target_ret in target_returns:
            try:
                weights = self.optimize_portfolio(target='target_return', target_return=target_ret)
                if weights is not None:
                    ret, vol, sharpe = self.portfolio_performance(weights)
                    frontier_returns.append(ret)
                    frontier_volatilities.append(vol)
                    frontier_sharpes.append(sharpe)
            except:
                continue
        
        return pd.DataFrame({
            'Return': frontier_returns,
            'Volatility': frontier_volatilities,
            'Sharpe': frontier_sharpes
        })
    
    def plot_efficient_frontier(self, show_assets=True):
        """Visualize the efficient frontier."""
        # Generate frontier
        frontier = self.generate_efficient_frontier()
        
        # Get optimal portfolios
        weights_max_sharpe = self.optimize_portfolio(target='max_sharpe')
        weights_min_vol = self.optimize_portfolio(target='min_volatility')
        
        ret_max_sharpe, vol_max_sharpe, sharpe_max_sharpe = self.portfolio_performance(weights_max_sharpe)
        ret_min_vol, vol_min_vol, sharpe_min_vol = self.portfolio_performance(weights_min_vol)
        
        # Plot
        plt.figure(figsize=(12, 7))
        
        # Efficient frontier
        plt.plot(frontier['Volatility'], frontier['Return'], 'b-', linewidth=2, label='Efficient Frontier')
        
        # Optimal portfolios
        plt.scatter(vol_max_sharpe, ret_max_sharpe, marker='*', s=500, c='gold', 
                   edgecolors='black', label=f'Max Sharpe (SR={sharpe_max_sharpe:.2f})', zorder=5)
        plt.scatter(vol_min_vol, ret_min_vol, marker='*', s=500, c='red',
                   edgecolors='black', label=f'Min Volatility', zorder=5)
        
        # Individual assets
        if show_assets:
            for i, ticker in enumerate(self.tickers):
                plt.scatter(np.sqrt(self.cov_matrix.iloc[i, i]), 
                          self.mean_returns.iloc[i],
                          marker='o', s=100, label=ticker)
        
        plt.xlabel('Volatility (Risk)', fontsize=12)
        plt.ylabel('Expected Return', fontsize=12)
        plt.title('Markowitz Efficient Frontier', fontsize=14, fontweight='bold')
        plt.legend(loc='best', fontsize=10)
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
    
    def display_optimal_portfolio(self, portfolio_type='max_sharpe'):
        """Display optimal portfolio allocation and metrics."""
        if portfolio_type == 'max_sharpe':
            weights = self.optimize_portfolio(target='max_sharpe')
            title = "MAXIMUM SHARPE RATIO PORTFOLIO"
        else:
            weights = self.optimize_portfolio(target='min_volatility')
            title = "MINIMUM VOLATILITY PORTFOLIO"
        
        if weights is None:
            print("Optimization failed")
            return
        
        ret, vol, sharpe = self.portfolio_performance(weights)
        
        print(f"\n{'='*70}")
        print(f"{title}")
        print(f"{'='*70}\n")
        
        print(f"Portfolio Metrics:")
        print(f"  Expected Annual Return: {ret*100:.2f}%")
        print(f"  Annual Volatility (Risk): {vol*100:.2f}%")
        print(f"  Sharpe Ratio: {sharpe:.3f}")
        print(f"\nOptimal Allocation:")
        
        allocation_df = pd.DataFrame({
            'Ticker': self.tickers,
            'Weight (%)': weights * 100,
            'Expected Return': self.mean_returns * 100
        }).sort_values('Weight (%)', ascending=False)
        
        print(allocation_df.to_string(index=False))
        
        # Investment calculator
        print(f"\n{'='*70}")
        print("Investment Amount Calculator")
        print(f"{'='*70}\n")
        
        investment = float(input("Enter total investment amount ($): "))
        
        print(f"\nFor ${investment:,.2f} investment:\n")
        for ticker, weight in zip(self.tickers, weights):
            amount = investment * weight
            print(f"  {ticker}: ${amount:,.2f} ({weight*100:.2f}%)")
        
        print(f"\n{'='*70}\n")
        
        return weights, ret, vol, sharpe
    
    def compare_portfolios(self):
        """Compare different portfolio strategies."""
        # Equal weight portfolio
        equal_weights = np.array([1/self.n_assets] * self.n_assets)
        eq_ret, eq_vol, eq_sharpe = self.portfolio_performance(equal_weights)
        
        # Max Sharpe portfolio
        max_sharpe_weights = self.optimize_portfolio(target='max_sharpe')
        ms_ret, ms_vol, ms_sharpe = self.portfolio_performance(max_sharpe_weights)
        
        # Min volatility portfolio
        min_vol_weights = self.optimize_portfolio(target='min_volatility')
        mv_ret, mv_vol, mv_sharpe = self.portfolio_performance(min_vol_weights)
        
        comparison = pd.DataFrame({
            'Strategy': ['Equal Weight', 'Max Sharpe', 'Min Volatility'],
            'Return (%)': [eq_ret*100, ms_ret*100, mv_ret*100],
            'Volatility (%)': [eq_vol*100, ms_vol*100, mv_vol*100],
            'Sharpe Ratio': [eq_sharpe, ms_sharpe, mv_sharpe]
        })
        
        print(f"\n{'='*70}")
        print("PORTFOLIO STRATEGY COMPARISON")
        print(f"{'='*70}\n")
        print(comparison.to_string(index=False))
        print(f"\n{'='*70}\n")
        
        return comparison


# Example Usage
if __name__ == "__main__":
    # Define your portfolio tickers
    tickers = ['AAPL', 'MSFT', 'GOOGL', 'JPM', 'JNJ']
    
    # Initialize optimizer
    optimizer = MarkowitzPortfolioOptimizer(
        tickers=tickers,
        period='3y',  # 3 years of historical data
        risk_free_rate=0.04  # 4% risk-free rate
    )
    
    # Fetch data from Yahoo Finance
    if optimizer.fetch_data():
        
        # Display individual asset statistics
        print("\nIndividual Asset Statistics (Annualized):")
        stats = pd.DataFrame({
            'Ticker': tickers,
            'Expected Return (%)': optimizer.mean_returns * 100,
            'Volatility (%)': np.sqrt(np.diag(optimizer.cov_matrix)) * 100
        })
        print(stats.to_string(index=False))
        
        # Compare different strategies
        optimizer.compare_portfolios()
        
        # Display optimal portfolio with investment calculator
        optimizer.display_optimal_portfolio(portfolio_type='min_volatility')
        
        # Plot efficient frontier
        optimizer.plot_efficient_frontier()

# 2. Black-Litterman Portfolio Optimization

## Overview

This section implements the **Black-Litterman model**, a Bayesian extension of the Markowitz framework widely used in institutional portfolio management.

Unlike traditional mean-variance optimization, which relies solely on historical estimates of expected returns, Black-Litterman combines:

- **Market equilibrium returns (implied by capitalization weights)**
- **Investor views (subjective or tactical forecasts)**
- **Uncertainty calibration around those views**

This produces a more stable and economically grounded set of expected returns.

## Methodology

The model is structured in three steps:

### 1. Market Equilibrium (Prior)

Implied returns are derived via reverse optimization:

- Market portfolio weights are based on market capitalizations  
- Risk aversion parameter controls equilibrium scaling  
- Returns are computed as:

  - Equilibrium returns = risk aversion × covariance matrix × market weights  

This represents the **neutral starting point of the market**.

---

### 2. Investor Views (Bayesian Inputs)

Investor beliefs are incorporated through:

- **P matrix** → defines relative or absolute views on assets  
- **Q vector** → expected return associated with each view  
- **Confidence parameter (τ, Ω)** → controls uncertainty around views  

This allows the model to express:

- Relative outperformance (e.g. AAPL > MSFT)
- Absolute return expectations (e.g. GOOGL = 8%)
- Multi-asset macro views

---

### 3. Posterior Estimation

The model combines prior and views using Bayesian updating to produce:

- **Posterior expected returns**
- **Updated covariance structure**

This results in a return distribution that is:

- Anchored to market equilibrium  
- Adjusted for investor convictions  
- Regularized to reduce estimation noise  

## Optimization

The posterior returns are then used in a standard constrained optimization framework:

- Maximize Sharpe ratio (default)
- Full investment constraint (weights sum to 1)
- Long-only constraint (no short selling)

## Outputs

The model produces:

- Market vs optimized portfolio weights  
- Equilibrium vs posterior expected returns  
- Portfolio risk/return metrics  
- Visual comparison of allocation shifts  

## Interpretation

Black-Litterman is designed to:

- Solve instability issues of Markowitz optimization  
- Incorporate forward-looking views in a disciplined way  
- Prevent extreme allocations driven by noisy historical returns  
- Anchor portfolios to a realistic market benchmark  

## Key Insight

The core advantage of this framework is that:

> Portfolio optimization becomes a **controlled deviation from the market portfolio**, rather than a purely statistical fit to historical data.

This makes it particularly suitable for institutional portfolio construction where stability and interpretability are as important as performance.

In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
from scipy.optimize import minimize

class BlackLittermanOptimizer:
    """
    Black-Litterman Portfolio Optimization. 
    Extends Markowitz by combining market equilibrium with investor views.
    """

    def __init__(self, tickers, market_caps=None, period='3y', risk_free_rate=0.04, tau=0.05):
        """
        Parameters:
        - tickers       : list of tickers
        - market_caps   : dict of {ticker: market_cap}. If None, fetched automatically.
        - period        : historical data period
        - risk_free_rate: annual risk-free rate
        - tau           : uncertainty in prior (typically 0.01-0.10)
        """
        self.tickers         = tickers
        self.n_assets        = len(tickers)
        self.period          = period
        self.risk_free_rate  = risk_free_rate
        self.tau             = tau
        self.market_caps     = market_caps

        # Data storage
        self.prices          = None
        self.returns         = None
        self.cov_matrix      = None
        self.market_weights  = None
        self.pi              = None   # equilibrium returns
        self.bl_returns      = None   # posterior BL returns
        self.bl_cov          = None   # posterior BL covariance
        self.optimal_weights = None

        print(f"Initializing Black-Litterman optimizer with {self.n_assets} aseets: {', '.join(tickers)}")

    # ============================================================
    # SECTION 1: DATA
    # ============================================================

    def fetch_data(self):
        """Fetch historical prices and market caps from Yahoo Finance."""
        print(f"\nFetching {self.period} of historical data...")

        try:
            data = yf.download(self.tickers, period=self.period, progress=False, auto_adjust=True)

            if len(self.tickers) == 1:
                self.prices = data[['Close']].copy()
                self.prices.columns = self.tickers
            else:
                self.prices = data['Close'].copy()

            self.prices  = self.prices.dropna()
            self.returns = self.prices.pct_change().dropna()
            self.cov_matrix = self.returns.cov() * 252

            print(f"✓ Data fetched: {self.prices.index[0].date()} to {self.prices.index[-1].date()}")

            # Fetch market caps if not provided
            if self.market_caps is None:
                print("Fetching market caps...")
                self.market_caps = {}
                for ticker in self.tickers:
                    try:
                        info = yf.Ticker(ticker).info
                        self.market_caps[ticker] = info.get('marketCap', 1e9)
                    except:
                        self.market_caps[ticker] = 1e9  # fallback

            # Compute market cap weights
            total_mc = sum(self.market_caps.values())
            self.market_weights = np.array([self.market_caps[t] / total_mc for t in self.tickers])

            print("\nMarket Cap Weights:")
            for t, w in zip(self.tickers, self.market_weights):
                print(f"  {t}: {w*100:.2f}%")

            return True

        except Exception as e:
            print(f"✗ Error: {e}")
            return False

    # ============================================================
    # SECTION 2: BLACK-LITTERMAN CORE
    # ============================================================

    def compute_equilibrium_returns(self, delta=2.5):
        """
        Compute implied equilibrium returns (Pi) using reverse optimization.
        Pi = delta * Sigma * w_market
        delta: risk aversion coefficient (typically 2.5)
        """
        self.delta = delta
        self.pi = delta * self.cov_matrix.values @ self.market_weights
        print("\nEquilibrium Returns (Pi):")
        for t, r in zip(self.tickers, self.pi):
            print(f"  {t}: {r*100:.2f}%")
        return self.pi

    def set_views(self, P, Q, omega=None, confidence=0.5):
        """
        Set investor views.

        Parameters:
        - P          : views matrix (k x n), each row is a view
                       e.g. [1, -1, 0, 0, 0] means asset 1 outperforms asset 2
                       e.g. [1,  0, 0, 0, 0] means absolute view on asset 1
        - Q          : views vector (k,), expected return for each view
                       e.g. [0.02] means "outperform by 2%"
        - omega      : uncertainty matrix (k x k). If None, computed from confidence.
        - confidence : scalar 0-1, used to auto-compute omega if not provided
                       higher = more confident in views vs equilibrium
        """
        self.P = np.array(P, dtype=float)
        self.Q = np.array(Q, dtype=float)

        if omega is None:
            # Auto-compute omega from confidence
            # High confidence -> small omega -> views dominate
            variance = (1 - confidence) / confidence
            self.omega = np.diag(
                [variance * float(self.P[i] @ self.cov_matrix.values @ self.P[i])
                 for i in range(len(Q))]
            )
        else:
            self.omega = np.array(omega)

        print(f"\nViews set: {len(Q)} view(s)")
        for i, (p, q) in enumerate(zip(self.P, self.Q)):
            assets_in_view = [self.tickers[j] for j in range(self.n_assets) if p[j] != 0]
            print(f"  View {i+1}: {assets_in_view} → {q*100:.2f}%")

    def compute_bl_returns(self):
        """
        Compute Black-Litterman posterior returns and covariance.

        BL Formula:
        mu_BL = [(tau*Sigma)^-1 + P'*Omega^-1*P]^-1 * [(tau*Sigma)^-1*Pi + P'*Omega^-1*Q]
        """
        Sigma     = self.cov_matrix.values
        tau       = self.tau
        Pi        = self.pi
        P         = self.P
        Q         = self.Q
        Omega     = self.omega

        # Prior precision
        tau_sigma_inv = np.linalg.inv(tau * Sigma)

        # Posterior precision
        omega_inv     = np.linalg.inv(Omega)
        M_inv         = tau_sigma_inv + P.T @ omega_inv @ P

        # Posterior mean (BL returns)
        bl_mean       = np.linalg.inv(M_inv) @ (tau_sigma_inv @ Pi + P.T @ omega_inv @ Q)

        # Posterior covariance
        bl_cov        = Sigma + np.linalg.inv(M_inv)

        self.bl_returns = pd.Series(bl_mean, index=self.tickers)
        self.bl_cov     = pd.DataFrame(bl_cov, index=self.tickers, columns=self.tickers)

        print("\nBlack-Litterman Posterior Returns:")
        for t, r in zip(self.tickers, self.bl_returns):
            eq_r = self.pi[self.tickers.index(t)]
            diff = (r - eq_r) * 100
            arrow = "↑" if diff > 0 else "↓"
            print(f"  {t}: {r*100:.2f}%  (vs equilibrium {eq_r*100:.2f}%  {arrow}{abs(diff):.2f}%)")

        return self.bl_returns, self.bl_cov

    # ============================================================
    # SECTION 3: OPTIMIZATION
    # ============================================================

    def portfolio_performance(self, weights, use_bl=True):
        """Calculate portfolio return, volatility, Sharpe."""
        weights = np.array(weights)
        ret     = np.sum((self.bl_returns if use_bl else self.pi) * weights)
        vol     = np.sqrt(weights @ self.bl_cov.values @ weights) if use_bl \
                  else np.sqrt(weights @ self.cov_matrix.values @ weights)
        sharpe  = (ret - self.risk_free_rate) / vol
        return ret, vol, sharpe

    def optimize(self, target='max_sharpe'):
        """Optimize portfolio using BL posterior returns."""
        constraints = [{'type': 'eq', 'fun': lambda x: np.sum(x) - 1}]
        bounds      = tuple((0, 1) for _ in range(self.n_assets))
        w0          = self.market_weights  # start from market weights (natural BL prior)

        if target == 'max_sharpe':
            obj = lambda w: -self.portfolio_performance(w)[2]
        else:
            obj = lambda w:  self.portfolio_performance(w)[1]

        result = minimize(obj, w0, method='SLSQP', bounds=bounds, constraints=constraints)

        if result.success:
            self.optimal_weights = result.x
            return result.x
        else:
            print(f"Optimization failed: {result.message}")
            return None

    # ============================================================
    # SECTION 4: RESULTS & VISUALIZATION
    # ============================================================

    def display_results(self):
        """Display full BL portfolio results."""
        weights = self.optimize(target='max_sharpe')
        if weights is None:
            return

        ret, vol, sharpe = self.portfolio_performance(weights)

        print(f"\n{'='*65}")
        print("  BLACK-LITTERMAN OPTIMAL PORTFOLIO")
        print(f"{'='*65}")
        print(f"  Expected Annual Return : {ret*100:.2f}%")
        print(f"  Annual Volatility      : {vol*100:.2f}%")
        print(f"  Sharpe Ratio           : {sharpe:.3f}")

        print(f"\n  {'Ticker':<8} {'BL Weight':>10} {'Mkt Weight':>12} {'Difference':>12} {'BL Return':>12}")
        print(f"  {'-'*56}")
        for i, t in enumerate(self.tickers):
            diff = (weights[i] - self.market_weights[i]) * 100
            arrow = "↑" if diff > 0 else "↓"
            print(f"  {t:<8} {weights[i]*100:>9.2f}%  {self.market_weights[i]*100:>10.2f}%  "
                  f"  {arrow}{abs(diff):>8.2f}%  {self.bl_returns[t]*100:>10.2f}%")

    def plot_results(self):
        """Plot BL weights vs market weights and return comparison."""
        weights = self.optimize(target='max_sharpe')
        if weights is None:
            return

        fig, axes = plt.subplots(1, 2, figsize=(14, 5))

        # Plot 1: Weights comparison
        x     = np.arange(self.n_assets)
        width = 0.35
        axes[0].bar(x - width/2, self.market_weights * 100, width, label='Market Weights', color='steelblue', alpha=0.8)
        axes[0].bar(x + width/2, weights * 100,             width, label='BL Weights',     color='coral',     alpha=0.8)
        axes[0].set_xticks(x)
        axes[0].set_xticklabels(self.tickers)
        axes[0].set_ylabel("Weight (%)")
        axes[0].set_title("BL Weights vs Market Cap Weights")
        axes[0].legend()
        axes[0].grid(True, alpha=0.3)

        # Plot 2: Equilibrium vs BL returns
        axes[1].bar(x - width/2, self.pi * 100,              width, label='Equilibrium Returns', color='steelblue', alpha=0.8)
        axes[1].bar(x + width/2, self.bl_returns.values * 100, width, label='BL Returns',          color='coral',     alpha=0.8)
        axes[1].set_xticks(x)
        axes[1].set_xticklabels(self.tickers)
        axes[1].set_ylabel("Expected Return (%)")
        axes[1].set_title("Equilibrium vs BL Posterior Returns")
        axes[1].legend()
        axes[1].grid(True, alpha=0.3)

        plt.suptitle("Black-Litterman Portfolio Analysis", fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.show()


# ============================================================
# EXAMPLE USAGE
# ============================================================

if __name__ == "__main__":

    tickers = ['AAPL', 'MSFT', 'GOOGL', 'JPM', 'JNJ']

    bl = BlackLittermanOptimizer(
        tickers=tickers,
        period='3y',
        risk_free_rate=0.04,
        tau=0.05
    )

    if bl.fetch_data():

        # Step 1: compute equilibrium returns from market cap weights
        bl.compute_equilibrium_returns(delta=2.5)

        # --------------------------------------------------------
        # Step 2: SET YOUR VIEWS HERE
        # --------------------------------------------------------
        # View 1: AAPL will outperform MSFT by 3% (relative view)
        # View 2: GOOGL will return 8% absolutely (absolute view)
        # View 3: JPM will outperform JNJ by 2% (relative view)

        P = [
            [ 1, -1,  0,  0,  0],   # AAPL outperforms MSFT
            [ 0,  0,  1,  0,  0],   # GOOGL absolute view
            [ 0,  0,  0,  1, -1],   # JPM outperforms JNJ
        ]
        Q = [0.03, 0.08, 0.02]      # by 3%, 8% absolute, by 2%

        bl.set_views(P, Q, confidence=0.6)  # 60% confident in views
        # --------------------------------------------------------

        # Step 3: compute BL posterior returns
        bl.compute_bl_returns()

        # Step 4: display and plot
        bl.display_results()
        bl.plot_results()

# 3. Hierarchical Risk Parity (HRP)

## Overview

This section implements the **Hierarchical Risk Parity (HRP)** framework, a modern portfolio construction method developed by Marcos López de Prado.

Unlike traditional optimization approaches, HRP does not rely on expected returns or mean-variance assumptions. Instead, it builds portfolios based on the **hierarchical structure of asset correlations**, allocating risk in a data-driven and stable manner.

This makes HRP particularly attractive in institutional settings where:
- Return estimates are unstable or unreliable  
- Covariance matrices are noisy or ill-conditioned  
- Robust diversification is preferred over return maximization  

## Methodology

The HRP process follows three key steps:

### 1. Distance Transformation

The correlation matrix is transformed into a distance metric:

- Highly correlated assets → small distance  
- Uncorrelated assets → larger distance  
- This converts the correlation structure into a geometric space suitable for clustering  

---

### 2. Hierarchical Clustering

Assets are grouped using hierarchical clustering (Ward linkage):

- Similar assets are grouped into clusters  
- A dendrogram structure is created  
- Assets are reordered based on hierarchical similarity  

This step identifies natural **risk clusters** in the portfolio.

---

### 3. Recursive Bisection Allocation

Portfolio weights are allocated using recursive splitting:

- Clusters are recursively divided into sub-clusters  
- Risk is allocated inversely proportional to cluster variance  
- Riskier clusters receive lower capital allocation  

This produces a fully diversified portfolio without requiring expected returns.

## Key Properties

HRP has several structural advantages:

- No reliance on return forecasts  
- No matrix inversion instability  
- Naturally diversified across correlation clusters  
- Stable out-of-sample behavior  
- Robust to estimation error  

## Outputs

The model generates:

- Hierarchical clustering structure (dendrogram)  
- Cluster-sorted correlation matrix  
- Portfolio weights based on risk allocation  
- Risk contribution decomposition by asset  
- Comparative performance vs Markowitz-style portfolios  

## Interpretation

HRP can be interpreted as:

> A **pure risk allocation framework** that replaces return optimization with structural diversification.

It ensures that:

- No single correlated group dominates portfolio risk  
- Risk is evenly distributed across independent sources  
- Portfolio construction is stable and reproducible  

## Extensions

This implementation further extends HRP with:

- Risk contribution diagnostics  
- Constrained weight allocation (min/max bounds)  
- Sharpe-based weighting adjustments  
- Portfolio stress and efficiency diagnostics  

These enhancements make the model closer to an **institutional risk management toolkit** rather than a purely academic clustering method.

In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.cluster.hierarchy import linkage, dendrogram, leaves_list
from scipy.spatial.distance import squareform

class HRPOptimizer:
    """
    Hierarchical Risk Parity (HRP) Portfolio Optimizer.
    Based on Marcos Lopez de Prado's methodology.
    Uses clustering to allocate risk across correlated asset groups.
    No return forecasts needed — purely data driven.
    """

    def __init__(self, tickers, period='3y', risk_free_rate=0.04):
        self.tickers         = tickers
        self.n_assets        = len(tickers)
        self.period          = period
        self.risk_free_rate  = risk_free_rate

        self.prices          = None
        self.returns         = None
        self.cov_matrix      = None
        self.corr_matrix     = None
        self.clusters        = None
        self.sorted_tickers  = None
        self.hrp_weights     = None

        print(f"Initializing HRP optimizer with {self.n_assets} assets: {', '.join(tickers)}")

    # ============================================================
    # SECTION 1: DATA
    # ============================================================

    def fetch_data(self):
        print(f"\nFetching {self.period} of historical data...")
        try:
            data = yf.download(self.tickers, period=self.period, progress=False, auto_adjust=True)
            self.prices     = data['Close'].dropna()
            self.returns    = self.prices.pct_change().dropna()
            self.cov_matrix = self.returns.cov() * 252
            self.corr_matrix= self.returns.corr()
            print(f"✓ Data fetched: {self.prices.index[0].date()} to {self.prices.index[-1].date()}")
            return True
        except Exception as e:
            print(f"✗ Error: {e}")
            return False

    # ============================================================
    # SECTION 2: HRP CORE — THREE STEPS
    # ============================================================

    def _corr_to_distance(self):
        """
        Step 1 — Convert correlation matrix to distance matrix.
        distance(i,j) = sqrt(0.5 * (1 - corr(i,j)))
        Perfectly correlated assets = distance 0
        Uncorrelated assets         = distance 0.707
        Perfectly anticorrelated    = distance 1
        """
        dist = np.sqrt(0.5 * (1 - self.corr_matrix))
        return dist

    def _cluster_assets(self):
        """
        Step 2 — Hierarchical clustering using Ward linkage.
        Groups similar (correlated) assets together.
        Returns linkage matrix and sorted ticker order.
        """
        dist_matrix  = self._corr_to_distance()
        
        # Convert to condensed distance matrix for scipy
        condensed    = squareform(dist_matrix.values, checks=False)
        
        # Ward linkage: minimizes variance within clusters
        self.linkage_matrix = linkage(condensed, method='ward')
        
        # Get the order of assets after clustering
        order               = leaves_list(self.linkage_matrix)
        self.sorted_tickers = [self.tickers[i] for i in order]
        self.sorted_indices  = order
        
        return self.linkage_matrix, self.sorted_tickers

    def _get_cluster_variance(self, cluster_items):
        """
        Compute variance of a cluster using inverse variance weighting.
        This is the building block of the recursive bisection.
        """
        cov_slice = self.cov_matrix.loc[cluster_items, cluster_items].values
        
        # Inverse variance weights within cluster
        inv_var   = 1 / np.diag(cov_slice)
        inv_var  /= inv_var.sum()
        
        # Cluster variance
        cluster_var = inv_var @ cov_slice @ inv_var
        return cluster_var

    def _recursive_bisection(self, sorted_items):
        """
        Step 3 — Recursive bisection allocation.
        Splits the sorted asset list in half recursively,
        allocating weight proportionally to inverse cluster variance.
        This ensures riskier clusters get less weight.
        """
        weights = pd.Series(1.0, index=sorted_items)
        
        # Stack of clusters to process
        clusters = [sorted_items]
        
        while clusters:
            # Split each cluster into two halves
            clusters = [
                subcluster
                for cluster in clusters
                for subcluster in [cluster[:len(cluster)//2],
                                   cluster[len(cluster)//2:]]
                if len(cluster) > 1
            ]
            
            # For each pair of adjacent clusters, allocate weights
            for i in range(0, len(clusters), 2):
                if i + 1 >= len(clusters):
                    break
                    
                left  = clusters[i]
                right = clusters[i+1]
                
                var_left  = self._get_cluster_variance(left)
                var_right = self._get_cluster_variance(right)
                
                # Allocate inversely proportional to variance
                # Riskier cluster gets less weight
                alpha = 1 - var_left / (var_left + var_right)
                
                weights[left]  *= alpha
                weights[right] *= 1 - alpha
        
        return weights

    def compute_hrp_weights(self):
        """Run the full HRP pipeline."""
        print("\nRunning HRP optimization...")
        print("  Step 1: Computing distance matrix from correlations...")
        
        print("  Step 2: Clustering assets hierarchically...")
        self._cluster_assets()
        print(f"           Asset order after clustering: {' → '.join(self.sorted_tickers)}")
        
        print("  Step 3: Recursive bisection allocation...")
        self.hrp_weights = self._recursive_bisection(self.sorted_tickers)
        
        # Reindex to original ticker order
        self.hrp_weights = self.hrp_weights[self.tickers]
        
        print("✓ HRP optimization complete")
        return self.hrp_weights

    # ============================================================
    # SECTION 3: PERFORMANCE METRICS
    # ============================================================

    def portfolio_performance(self, weights):
        w   = np.array([weights[t] for t in self.tickers])
        ret = (self.returns.mean() * 252) @ w
        vol = np.sqrt(w @ self.cov_matrix.values @ w)
        sharpe = (ret - self.risk_free_rate) / vol
        return ret, vol, sharpe

    def risk_contributions(self, weights):
        """
        Compute each asset's contribution to total portfolio risk.
        This is what HRP tries to make more balanced vs Markowitz.
        """
        w       = np.array([weights[t] for t in self.tickers])
        port_vol= np.sqrt(w @ self.cov_matrix.values @ w)
        marginal= self.cov_matrix.values @ w / port_vol
        contrib = w * marginal
        contrib_pct = contrib / contrib.sum()
        return pd.Series(contrib_pct, index=self.tickers)

    def compare_with_markowitz(self):
        """Compare HRP vs equal weight vs Markowitz min variance."""
        from scipy.optimize import minimize

        # Equal weight
        eq_w = pd.Series(1/self.n_assets, index=self.tickers)

        # Markowitz min variance
        constraints = [{'type': 'eq', 'fun': lambda x: np.sum(x) - 1}]
        bounds      = tuple((0, 1) for _ in range(self.n_assets))
        w0          = np.array([1/self.n_assets]*self.n_assets)
        result      = minimize(
            lambda w: np.sqrt(w @ self.cov_matrix.values @ w),
            w0, method='SLSQP', bounds=bounds, constraints=constraints
        )
        mv_w = pd.Series(result.x, index=self.tickers)

        # Metrics
        rows = []
        for name, w in [("Equal Weight", eq_w), ("Min Variance", mv_w), ("HRP", self.hrp_weights)]:
            ret, vol, sharpe = self.portfolio_performance(w)
            rc = self.risk_contributions(w)
            rows.append({
                'Strategy'         : name,
                'Return (%)'       : round(ret*100, 2),
                'Volatility (%)'   : round(vol*100, 2),
                'Sharpe Ratio'     : round(sharpe, 3),
                'Max Risk Contrib' : f"{rc.max()*100:.1f}% ({rc.idxmax()})",
                'Min Risk Contrib' : f"{rc.min()*100:.1f}% ({rc.idxmin()})"
            })

        comparison = pd.DataFrame(rows)
        print(f"\n{'='*75}")
        print("  STRATEGY COMPARISON")
        print(f"{'='*75}")
        print(comparison.to_string(index=False))
        return comparison, eq_w, mv_w

    # ============================================================
    # SECTION 4: VISUALIZATION
    # ============================================================

    def plot_dendrogram(self):
        """Plot the hierarchical clustering dendrogram."""
        fig, ax = plt.subplots(figsize=(10, 4))
        dendrogram(
            self.linkage_matrix,
            labels=self.tickers,
            ax=ax,
            color_threshold=0.7 * max(self.linkage_matrix[:,2])
        )
        ax.set_title("Asset Clustering Dendrogram", fontsize=13, fontweight='bold')
        ax.set_xlabel("Assets")
        ax.set_ylabel("Distance")
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

    def plot_correlation_matrix(self):
        """Plot correlation matrix sorted by cluster order."""
        sorted_corr = self.corr_matrix.loc[self.sorted_tickers, self.sorted_tickers]
        fig, axes   = plt.subplots(1, 2, figsize=(14, 5))

        for ax, corr, title in zip(
            axes,
            [self.corr_matrix, sorted_corr],
            ["Original Correlation Matrix", "Clustered Correlation Matrix"]
        ):
            im = ax.imshow(corr.values, cmap='RdYlGn', vmin=-1, vmax=1)
            ax.set_xticks(range(self.n_assets))
            ax.set_yticks(range(self.n_assets))
            labels = corr.columns.tolist()
            ax.set_xticklabels(labels, rotation=45)
            ax.set_yticklabels(labels)
            for i in range(self.n_assets):
                for j in range(self.n_assets):
                    ax.text(j, i, f"{corr.values[i,j]:.2f}",
                            ha='center', va='center', fontsize=8,
                            color='black')
            ax.set_title(title, fontweight='bold')
            plt.colorbar(im, ax=ax)

        plt.suptitle("Correlation Structure — Before and After Clustering", fontsize=13)
        plt.tight_layout()
        plt.show()

    def plot_full_results(self):
        """Master plot: weights, risk contributions, and comparison."""
        comparison, eq_w, mv_w = self.compare_with_markowitz()

        fig, axes = plt.subplots(1, 3, figsize=(18, 5))

        x     = np.arange(self.n_assets)
        width = 0.25

        # ---- Plot 1: Weights comparison ----
        axes[0].bar(x - width, eq_w.values * 100,           width, label='Equal Weight',  color='steelblue', alpha=0.85)
        axes[0].bar(x,         mv_w.values * 100,           width, label='Min Variance',  color='coral',     alpha=0.85)
        axes[0].bar(x + width, self.hrp_weights.values*100, width, label='HRP',           color='seagreen',  alpha=0.85)
        axes[0].set_xticks(x)
        axes[0].set_xticklabels(self.tickers, rotation=45)
        axes[0].set_ylabel("Weight (%)")
        axes[0].set_title("Portfolio Weights Comparison")
        axes[0].legend()
        axes[0].grid(True, alpha=0.3)

        # ---- Plot 2: Risk contributions ----
        rc_eq  = self.risk_contributions(eq_w)
        rc_mv  = self.risk_contributions(mv_w)
        rc_hrp = self.risk_contributions(self.hrp_weights)

        axes[1].bar(x - width, rc_eq.values  * 100, width, label='Equal Weight', color='steelblue', alpha=0.85)
        axes[1].bar(x,         rc_mv.values  * 100, width, label='Min Variance', color='coral',     alpha=0.85)
        axes[1].bar(x + width, rc_hrp.values * 100, width, label='HRP',          color='seagreen',  alpha=0.85)
        axes[1].axhline(100/self.n_assets, color='black', linestyle='--',
                        linewidth=1.2, label=f'Equal contrib ({100/self.n_assets:.1f}%)')
        axes[1].set_xticks(x)
        axes[1].set_xticklabels(self.tickers, rotation=45)
        axes[1].set_ylabel("Risk Contribution (%)")
        axes[1].set_title("Risk Contributions by Asset")
        axes[1].legend()
        axes[1].grid(True, alpha=0.3)

        # ---- Plot 3: Metrics bar chart ----
        metrics   = ['Return (%)', 'Volatility (%)', 'Sharpe Ratio']
        strategies= comparison['Strategy'].tolist()
        colors    = ['steelblue', 'coral', 'seagreen']
        bar_width = 0.2
        x3        = np.arange(len(metrics))

        for i, (strategy, color) in enumerate(zip(strategies, colors)):
            vals = [comparison.loc[comparison['Strategy']==strategy, m].values[0] for m in metrics]
            axes[2].bar(x3 + i*bar_width, vals, bar_width, label=strategy, color=color, alpha=0.85)

        axes[2].set_xticks(x3 + bar_width)
        axes[2].set_xticklabels(metrics)
        axes[2].set_title("Performance Metrics Comparison")
        axes[2].legend()
        axes[2].grid(True, alpha=0.3)

        plt.suptitle("HRP Portfolio Analysis", fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.show()

        return comparison

    def display_results(self):
        """Print full results summary."""
        ret, vol, sharpe = self.portfolio_performance(self.hrp_weights)
        rc               = self.risk_contributions(self.hrp_weights)

        print(f"\n{'='*60}")
        print("  HRP OPTIMAL PORTFOLIO")
        print(f"{'='*60}")
        print(f"  Expected Annual Return : {ret*100:.2f}%")
        print(f"  Annual Volatility      : {vol*100:.2f}%")
        print(f"  Sharpe Ratio           : {sharpe:.3f}")

        print(f"\n  {'Ticker':<8} {'Weight':>10} {'Risk Contrib':>14}")
        print(f"  {'-'*35}")
        for t in self.tickers:
            print(f"  {t:<8} {self.hrp_weights[t]*100:>9.2f}%  {rc[t]*100:>12.2f}%")

    def diagnose_portfolio(self):
            """
            Full diagnostic: identify which assets are dragging down
            return, inflating volatility, and how efficiently
            risk is being compensated.
            """
            import matplotlib.pyplot as plt
            import pandas as pd
            import numpy as np

            w        = self.hrp_weights
            ann_ret  = self.returns.mean() * 252
            ann_vol  = pd.Series(np.sqrt(np.diag(self.cov_matrix.values)), index=self.tickers)
            rc       = self.risk_contributions(w)

            # ---- Return drag ----
            # How much each asset contributes to total portfolio return
            return_contrib = pd.Series(
                {t: w[t] * ann_ret[t] for t in self.tickers}
                )

            # ---- Sharpe per asset ----
            asset_sharpe = (ann_ret - self.risk_free_rate) / ann_vol

            # ---- Return per unit of risk contributed ----
            # If an asset contributes a lot of risk but little return -> drag
            return_per_risk = return_contrib / rc

            # ---- Diagnostic table ----
            diag = pd.DataFrame({
                'Weight (%)'        : (w * 100).round(2),
                'Asset Return (%)'  : (ann_ret * 100).round(2),
                'Asset Vol (%)'     : (ann_vol * 100).round(2),
                'Asset Sharpe'      : asset_sharpe.round(3),
                'Return Contrib (%)': (return_contrib * 100).round(3),
                'Risk Contrib (%)'  : (rc * 100).round(2),
                'Return/Risk'       : return_per_risk.round(3),
            })

            # ---- Flag draggers ----
            # An asset is a drag if its return contribution is negative
            # OR if its return/risk ratio is below the portfolio average
            port_ret, port_vol, port_sharpe = self.portfolio_performance(w)
            avg_return_per_risk = return_per_risk.mean()
        
            diag['Status'] = diag.apply(lambda row:
                '🔴 DRAG'     if row['Return Contrib (%)'] < 0 else
                '🟡 WEAK'     if row['Return/Risk'] < avg_return_per_risk else
                '🟢 OK',
                axis=1
            )
        
            diag = diag.sort_values('Return/Risk')
        
            print(f"\n{'='*80}")
            print("  HRP PORTFOLIO DIAGNOSTIC")
            print(f"{'='*80}")
            print(f"  Portfolio Return    : {port_ret*100:.2f}%")
            print(f"  Portfolio Volatility: {port_vol*100:.2f}%")
            print(f"  Portfolio Sharpe    : {port_sharpe:.3f}")
            print(f"\n  Avg Return/Risk Ratio: {avg_return_per_risk:.3f}")
            print(f"\n{'='*80}")
            print(diag.to_string())
            print(f"\n  🔴 DRAG  = negative return contribution")
            print(f"  🟡 WEAK  = below average return per unit of risk")
            print(f"  🟢 OK    = above average return per unit of risk")
        
            # ---- Plot ----
            fig, axes = plt.subplots(1, 3, figsize=(16, 5))
        
            colors = ['red' if v < 0 else 'steelblue' for v in return_contrib.values]
        
            # Plot 1: Return contributions
            axes[0].barh(self.tickers, return_contrib.values * 100, color=colors, alpha=0.85)
            axes[0].axvline(0, color='black', linewidth=1)
            axes[0].set_xlabel("Return Contribution (%)")
            axes[0].set_title("Return Contribution by Asset")
            axes[0].grid(True, alpha=0.3)
        
            # Plot 2: Return vs Risk contribution scatter
            for t in self.tickers:
                color = 'red'   if return_contrib[t] < 0 else \
                        'orange' if return_per_risk[t] < avg_return_per_risk else 'seagreen'
                axes[1].scatter(rc[t]*100, return_contrib[t]*100, s=200, color=color, zorder=5)
                axes[1].annotate(t, (rc[t]*100, return_contrib[t]*100),
                                 textcoords="offset points", xytext=(8, 4), fontsize=9)
        
            axes[1].axhline(0, color='black', linewidth=0.8, linestyle='--')
            axes[1].set_xlabel("Risk Contribution (%)")
            axes[1].set_ylabel("Return Contribution (%)")
            axes[1].set_title("Return vs Risk Contribution\n(top-right = efficient)")
            axes[1].grid(True, alpha=0.3)
        
            # Add quadrant labels
            xlim = axes[1].get_xlim()
            ylim = axes[1].get_ylim()
            axes[1].text(xlim[1]*0.6, ylim[1]*0.85, "High risk\nHigh return", fontsize=7, color='gray')
            axes[1].text(xlim[0],     ylim[1]*0.85, "Low risk\nHigh return",  fontsize=7, color='gray')
            axes[1].text(xlim[1]*0.6, ylim[0]*0.85, "High risk\nLow return",  fontsize=7, color='red', alpha=0.6)
            axes[1].text(xlim[0],     ylim[0]*0.85, "Low risk\nLow return",   fontsize=7, color='gray')
        
            # Plot 3: Asset Sharpe ratios
            sharpe_colors = ['red' if s < 0 else 'steelblue' for s in asset_sharpe.values]
            axes[2].barh(self.tickers, asset_sharpe.values, color=sharpe_colors, alpha=0.85)
            axes[2].axvline(port_sharpe, color='black', linestyle='--',
                            linewidth=1.5, label=f'Portfolio Sharpe ({port_sharpe:.2f})')
            axes[2].axvline(0, color='red', linewidth=0.8)
            axes[2].set_xlabel("Sharpe Ratio")
            axes[2].set_title("Individual Asset Sharpe Ratios\nvs Portfolio Sharpe")
            axes[2].legend(fontsize=8)
            axes[2].grid(True, alpha=0.3)
        
            plt.suptitle("HRP Portfolio Diagnostic — Drag Analysis", fontsize=13, fontweight='bold')
            plt.tight_layout()
            plt.show()
        
            return diag

    def compute_hrp_weights_constrained(self, 
                                     min_weight=0.02, 
                                     max_weight=0.40,
                                     sharpe_floor=None,
                                     sharpe_floor_weight=0.05):
            """
            HRP with weight constraints.
            
            Parameters:
            - min_weight         : minimum weight for any asset (e.g. 0.02 = 2%)
            - max_weight         : maximum weight for any asset (e.g. 0.40 = 40%)
            - sharpe_floor       : assets with Sharpe above this get a minimum weight
                                   of sharpe_floor_weight (e.g. 1.0 means Sharpe > 1)
            - sharpe_floor_weight: minimum weight guaranteed to high-Sharpe assets
            """
            # Start from raw HRP weights
            weights = self.compute_hrp_weights().copy()
        
            ann_ret     = self.returns.mean() * 252
            ann_vol     = pd.Series(np.sqrt(np.diag(self.cov_matrix.values)), index=self.tickers)
            asset_sharpe= (ann_ret - self.risk_free_rate) / ann_vol
        
            print(f"\n--- Applying Constraints ---")
            print(f"  Min weight          : {min_weight*100:.1f}%")
            print(f"  Max weight          : {max_weight*100:.1f}%")
            if sharpe_floor:
                print(f"  Sharpe floor        : {sharpe_floor} → min weight {sharpe_floor_weight*100:.1f}%")
        
            # ---- Step 1: Apply Sharpe floor ----
            # High quality assets guaranteed a minimum weight
            if sharpe_floor is not None:
                for t in self.tickers:
                    if asset_sharpe[t] > sharpe_floor:
                        if weights[t] < sharpe_floor_weight:
                            print(f"  ↑ {t}: Sharpe={asset_sharpe[t]:.2f} → "
                                  f"floor applied {weights[t]*100:.2f}% → {sharpe_floor_weight*100:.2f}%")
                            weights[t] = sharpe_floor_weight
        
            # ---- Step 2: Apply min/max bounds ----
            for t in self.tickers:
                if weights[t] < min_weight:
                    print(f"  ↑ {t}: {weights[t]*100:.2f}% → min {min_weight*100:.2f}%")
                    weights[t] = min_weight
                elif weights[t] > max_weight:
                    print(f"  ↓ {t}: {weights[t]*100:.2f}% → max {max_weight*100:.2f}%")
                    weights[t] = max_weight
        
            # ---- Step 3: Renormalize to sum to 1 ----
            weights = weights / weights.sum()
        
            self.hrp_weights_constrained = weights
        
            # ---- Compare raw vs constrained ----
            ret_raw,  vol_raw,  sr_raw  = self.portfolio_performance(self.hrp_weights)
            ret_con,  vol_con,  sr_con  = self.portfolio_performance(weights)
        
            print(f"\n{'='*60}")
            print(f"  {'Metric':<25} {'Raw HRP':>12} {'Constrained':>12}")
            print(f"  {'-'*50}")
            print(f"  {'Return (%)':<25} {ret_raw*100:>11.2f}% {ret_con*100:>11.2f}%")
            print(f"  {'Volatility (%)':<25} {vol_raw*100:>11.2f}% {vol_con*100:>11.2f}%")
            print(f"  {'Sharpe Ratio':<25} {sr_raw:>12.3f} {sr_con:>12.3f}")
            print(f"{'='*60}")
        
            print(f"\n  {'Ticker':<8} {'Raw HRP':>10} {'Constrained':>12} {'Change':>10} {'Sharpe':>8}")
            print(f"  {'-'*52}")
            for t in self.tickers:
                diff = (weights[t] - self.hrp_weights[t]) * 100
                arrow = "↑" if diff > 0.01 else ("↓" if diff < -0.01 else "→")
                print(f"  {t:<8} {self.hrp_weights[t]*100:>9.2f}%  "
                      f"{weights[t]*100:>10.2f}%  "
                      f"{arrow}{abs(diff):>7.2f}%  "
                      f"{asset_sharpe[t]:>8.3f}")
        
            return weights
        
    def plot_constrained_vs_raw(self):
            """Compare raw HRP vs constrained HRP visually."""
            if not hasattr(self, 'hrp_weights_constrained'):
                print("Run compute_hrp_weights_constrained() first.")
                return
        
            ann_ret      = self.returns.mean() * 252
            ann_vol      = pd.Series(np.sqrt(np.diag(self.cov_matrix.values)), index=self.tickers)
            asset_sharpe = (ann_ret - self.risk_free_rate) / ann_vol
        
            rc_raw = self.risk_contributions(self.hrp_weights)
            rc_con = self.risk_contributions(self.hrp_weights_constrained)
        
            fig, axes = plt.subplots(1, 3, figsize=(18, 5))
            x     = np.arange(self.n_assets)
            width = 0.35
        
            # Plot 1: Weights
            axes[0].bar(x - width/2, self.hrp_weights.values * 100,
                        width, label='Raw HRP',        color='steelblue', alpha=0.85)
            axes[0].bar(x + width/2, self.hrp_weights_constrained.values * 100,
                        width, label='Constrained HRP', color='seagreen',  alpha=0.85)
            axes[0].set_xticks(x)
            axes[0].set_xticklabels(self.tickers, rotation=45)
            axes[0].set_ylabel("Weight (%)")
            axes[0].set_title("Weights: Raw vs Constrained HRP")
            axes[0].legend()
            axes[0].grid(True, alpha=0.3)
        
            # Plot 2: Risk contributions
            axes[1].bar(x - width/2, rc_raw.values * 100,
                        width, label='Raw HRP',        color='steelblue', alpha=0.85)
            axes[1].bar(x + width/2, rc_con.values * 100,
                        width, label='Constrained HRP', color='seagreen',  alpha=0.85)
            axes[1].axhline(100/self.n_assets, color='black', linestyle='--',
                            linewidth=1.2, label=f'Equal contrib ({100/self.n_assets:.1f}%)')
            axes[1].set_xticks(x)
            axes[1].set_xticklabels(self.tickers, rotation=45)
            axes[1].set_ylabel("Risk Contribution (%)")
            axes[1].set_title("Risk Contributions: Raw vs Constrained")
            axes[1].legend()
            axes[1].grid(True, alpha=0.3)
        
            # Plot 3: Sharpe vs weight scatter
            for t in self.tickers:
                axes[2].scatter(asset_sharpe[t], self.hrp_weights[t]*100,
                                color='steelblue', s=150, alpha=0.8)
                axes[2].scatter(asset_sharpe[t], self.hrp_weights_constrained[t]*100,
                                color='seagreen', s=150, alpha=0.8)
                axes[2].plot([asset_sharpe[t], asset_sharpe[t]],
                             [self.hrp_weights[t]*100, self.hrp_weights_constrained[t]*100],
                             color='gray', linewidth=1, linestyle='--')
                axes[2].annotate(t, (asset_sharpe[t], self.hrp_weights_constrained[t]*100),
                                 textcoords="offset points", xytext=(6, 4), fontsize=8)
        
            axes[2].set_xlabel("Asset Sharpe Ratio")
            axes[2].set_ylabel("Weight (%)")
            axes[2].set_title("Sharpe vs Weight\n(blue=raw, green=constrained)")
            axes[2].grid(True, alpha=0.3)
        
            blue_patch  = mpatches.Patch(color='steelblue', label='Raw HRP')
            green_patch = mpatches.Patch(color='seagreen',  label='Constrained HRP')
            axes[2].legend(handles=[blue_patch, green_patch])
        
            plt.suptitle("Raw vs Constrained HRP", fontsize=14, fontweight='bold')
            plt.tight_layout()
            plt.show()

# ============================================================
# EXAMPLE USAGE
# ============================================================

if __name__ == "__main__":

    tickers = ['NEE', 'SO', 'CEG', 'DUK', 'AEP', 'SRE', 'VST', 'D', 'EXC', 'XEL']

    hrp = HRPOptimizer(tickers=tickers, period='3y', risk_free_rate=0.04)

    if hrp.fetch_data():
        hrp.compute_hrp_weights()
        hrp.display_results()
        hrp.plot_dendrogram()
        hrp.plot_correlation_matrix()
        hrp.plot_full_results()
        hrp.diagnose_portfolio()

        # ---- Constrained version ----
        hrp.compute_hrp_weights_constrained(
            min_weight          = 0.02,   # no asset below 2%
            max_weight          = 0.35,   # no asset above 35%
            sharpe_floor        = 1.0,    # assets with Sharpe > 1 get at least...
            sharpe_floor_weight = 0.08   # ...8% minimum weight
        )
        hrp.plot_constrained_vs_raw()